# S3 · Modeling 30 daily balances per account

We fit each forecasting method to the same account histories. Before scoring, we clip every method's final predictions to ±€1,000, including the classical benchmarks, both Chronos models, the ensembles and the zero control. We preserve the observed balances.

## 1 · Two chronological comparisons

We use the prepared 500-account dataset and the two pinned checkpoints listed in [S2](S2__selecting_models.ipynb). Each evaluation forecast covers 30 days.

- **Original windows.** We select the policy using five origins from April through August 2022 and evaluate it in October, November, and December 2022. These windows informed the study design, so we report the result as a retrospective backtest.
- **Earlier replay.** We select the policy using five earlier origins in 2021 and evaluate it in five 30-day windows from December 2021 through April 2022.

In [1]:
import os
import sys
from importlib.util import find_spec
from pathlib import Path

import polars as pl
import yaml
from IPython.display import display

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "parameters.yml").is_file())
sys.path.insert(0, str(ROOT / "src"))

from bank_balance_forecasting.forecast_pilot import FOUNDATION_MODELS
from bank_balance_forecasting.modeling.diagnostics import diagnose_mape_outlier
from bank_balance_forecasting.modeling.evaluation import score_model_groups
from bank_balance_forecasting.modeling.forecasting import (
    forecast_cpu, forecast_foundations, forecast_submission,
)
from bank_balance_forecasting.modeling.group_caps import assign_origin_groups, compare_group_caps
from bank_balance_forecasting.modeling.threshold import (
    TARGET_MAPE, prepare_threshold_cases, select_threshold_policy,
)

DATASET, CUTOFF = "medium", "2022-12-31"
source = ROOT / "data" / "prepared" / DATASET / CUTOFF
balances = pl.read_parquet(source / "prepared_balances.parquet")
events = pl.read_parquet(source / "event_flags.parquet")
settings = yaml.safe_load((ROOT / "parameters.yml").read_text())
config = {**settings["stage3"], "foundation_models": list(FOUNDATION_MODELS)}
cap = config["forecast_cap"]
if find_spec("chronos") is None:
    raise ImportError("We require the chronos project extra to reproduce this study.")

cases = prepare_threshold_cases(balances)
display(cases.group_by("study").agg(
    pl.col("origin").n_unique().alias("Origins"),
    pl.col("account_id").n_unique().alias("Accounts"),
    pl.len().alias("Account-origins"),
).sort("study"))

study,Origins,Accounts,Account-origins
str,u32,u32,u32
"""Earlier chronological replay""",10,500,5000
"""Original 2022 windows""",8,500,4000


## 2 · Candidate forecasts and common policy selection

We apply the shared cap after building and calibrating the ensembles. We use the zero control as a benchmark. To select a history-based policy, we require a worst selection-window MAPE of at most 150% and a worst stress bound of at most 160%. The stress bound allows one account to spend seven consecutive days at a balance of magnitude €1. We select the eligible method with the lowest selection WAPE and lock it before evaluating the later windows.

### Actual forecasting methods

We use the labels below in the scorecards. We fit or apply each base method to each account's own history. For S3, we form ensembles and apply calibration before clipping the final predictions. The SMA, weekly persistence and drift rules use forward-filled history. The calendar regressions use up to 365 days of valid daily changes.

| Scorecard label | Forecasting method |
|---|---|
| `Last balance` | Persistence. We repeat the most recent observed balance for all 30 days. |
| `SMA 7` | 7-day simple moving average. We forecast one constant level from the last seven daily balances. |
| `SMA 14` | 14-day simple moving average. We forecast one constant level from the last 14 daily balances. |
| `SMA 28` | 28-day simple moving average. We forecast one constant level from the last 28 daily balances. |
| `Seasonal naive 7` | Weekly seasonal persistence. We cycle through the last seven daily balances. |
| `Local level` | Simple exponential smoothing. We fit a constant forecast level using up to 365 days and a fixed level gain of 0.20. |
| `Damped Holt` | Holt's damped trend method. We use up to 365 days, a level gain of 0.20, a trend gain of 0.05 and damping of 0.90. |
| `Drift 90` | Linear drift. We extrapolate the slope between the origin balance and the balance 90 days earlier. |
| `AR(1) changes` | First-order autoregression on daily balance changes. We forecast changes and accumulate them from the last level. |
| `Weekday changes` | Ridge regression on daily changes with weekday indicators. We accumulate the predicted changes. |
| `Calendar changes` | Ridge regression on daily changes with weekday indicators and an account-specific monthly inflow window when available. We accumulate the predicted changes. |
| `Two-part calendar` | Logistic regression for movement probability and ridge regression for signed size on movement days, using calendar features. We multiply the predictions and accumulate the expected changes. |
| `Chronos-Bolt Tiny` | Zero-shot median forecast from Amazon's pretrained `amazon/chronos-bolt-tiny` checkpoint. We use up to 512 daily history positions per account. |
| `Chronos-2 Small` | Zero-shot median forecast from the pretrained `autogluon/chronos-2-small` checkpoint. We use up to 512 daily history positions per account. |
| `Zero control` | Constant zero. We forecast zero on every day. |
| `MAPE median 30` | Flat weighted-median forecast from the last 30 calendar days. We weight finite, nonzero balances by the inverse of their absolute value. |
| `MAPE median 90` | Flat weighted-median forecast from the last 90 calendar days. We weight finite, nonzero balances by the inverse of their absolute value. |
| `MAPE median 365` | Flat weighted-median forecast from the last 365 calendar days. We weight finite, nonzero balances by the inverse of their absolute value. |
| `Mean classical` | Equal-weight mean of Last balance, SMA 14 and Local level. |
| `Mean MAPE medians` | Equal-weight mean of MAPE median 30, MAPE median 90 and MAPE median 365. |
| `Mean two Chronos` | Equal-weight mean of the two Chronos median forecasts. |
| `Mean Chronos and persistence` | Equal-weight mean of the two Chronos median forecasts and Last balance. |
| `Mean mixed` | Equal-weight mean of Last balance, Chronos-2 Small and MAPE median 365. |
| `Median mixed` | Pointwise median of Last balance, Chronos-2 Small and MAPE median 365. |
| `Chronos quantile pool (MAPE)` | We pool nine quantiles from each Chronos model and take a weighted median for each day. We weight each quantile by the inverse of its absolute value, with a €0.01 denominator floor. |
| `Calibrated MAPE median 365` | MAPE median 365 with the balance-scale calibration described below. |
| `Calibrated Mean MAPE medians` | Mean MAPE medians with balance-scale calibration. |
| `Calibrated Median mixed` | Median mixed with balance-scale calibration. |
| `Calibrated Chronos quantile pool (MAPE)` | Chronos quantile pool (MAPE) with balance-scale calibration. We used this ensemble for the January delivery after applying the shared cap. |

For **balance-scale calibration**, we group account-origins by the median absolute balance over the last 90 calendar days, with band boundaries at €100, €1,000 and €10,000. Within each band we fit a multiplier between 0 and 1 on the earlier selection windows to minimize MAPE. We apply the fitted multiplier on the later windows. We tune the final cap separately from these model definitions when comparing shared and group-specific caps.

In [2]:
classical = forecast_cpu(cases, config)
foundation = forecast_foundations(cases, config)
policy, selected, scores, by_origin, model_paths = select_threshold_policy(
    cases, classical, foundation, cap,
)
print(f"{scores['model'].n_unique()} candidates and controls · shared cap ±€{cap:,.0f}")
with pl.Config(tbl_cols=7, fmt_str_lengths=55):
    display(selected.select(
        "study", "model", "Worst selection MAPE (%)",
        "Worst selection stress (%)", "Selection WAPE (%)",
    ).with_columns(pl.selectors.float().round(2)))

study,model,Worst selection MAPE (%),Worst selection stress (%),Selection WAPE (%)
str,str,f64,f64,f64
"""Original 2022 windows""","""Calibrated Chronos quantile pool (MAPE)""",103.43,150.14,98.16
"""Earlier chronological replay""","""Calibrated Chronos quantile pool (MAPE)""",109.25,155.97,97.34


29 candidates and controls · shared cap ±€1,000


## 3 · Group-specific caps

We hold the chosen forecast fixed. At each origin, we fit the three S2 group roles using balances available through that date. We compare one shared cap with a separate cap for each group, using €500, €750, €1,000, €1,250 and €1,500 as candidates for both. We tune both policies under the same selection rules and apply the locked caps to the later windows. This isolates the contribution of group membership from the choice of forecasting model.

In [3]:
labels = assign_origin_groups(balances, cases, settings)
selected_paths = model_paths.filter(pl.col("Selected")).drop("Selected")
cap_scores, group_scores = compare_group_caps(cases, labels, selected_paths, cap)
display(cap_scores.select(
    "study", "policy", "G01", "G02", "G03", "MAPE (%)", "WAPE (%)",
).with_columns(pl.selectors.float().round(2)))

study,policy,G01,G02,G03,MAPE (%),WAPE (%)
str,str,f64,f64,f64,f64,f64
"""Original 2022 windows""","""Tuned shared cap""",1000.0,1000.0,1000.0,122.75,98.28
"""Original 2022 windows""","""Tuned group caps""",1250.0,1250.0,750.0,126.29,98.27
"""Earlier chronological replay""","""Tuned shared cap""",1000.0,1000.0,1000.0,96.27,97.53
"""Earlier chronological replay""","""Tuned group caps""",1000.0,1000.0,1000.0,96.27,97.53


## 4 · Evaluation exports and January forecasts

We save the aggregate study, window and group scores for [S4](S4__evaluation.ipynb). We trace the largest uncapped last-balance error to its source observations. For the submission, we use the shared-cap policy selected on the original study.

In [4]:
results = ROOT / "data" / "results" / DATASET / CUTOFF
results.mkdir(parents=True, exist_ok=True)
scores.write_json(results / "evaluation_scores.json")
model_groups, cap_effect = score_model_groups(cases, labels, model_paths, cap)
model_groups.write_json(results / "model_group_scores.json")
cap_effect.write_json(results / "selected_cap_effect.json")
by_origin.write_json(results / "evaluation_windows.json")
cap_scores.write_json(results / "cap_comparison.json")
group_scores.write_json(results / "group_cap_scores.json")
diagnose_mape_outlier(balances, events, cases, classical, cap).write_json(
    results / "mape_diagnostic.json"
)

chosen = by_origin.filter(pl.col("Selected"))
if chosen.filter(pl.col("MAPE (%)") > TARGET_MAPE).height:
    raise RuntimeError("The selected shared-cap policy failed at least one evaluation window")
submission = forecast_submission(balances, policy, settings["stage0"], settings["stage1"], config)
assert submission.shape == (balances["account_id"].n_unique(), 31)
submission.write_csv(results / "submission.csv")
print(f"{submission.height} accounts · {submission.columns[1]} to {submission.columns[-1]}")
print((results / "submission.csv").relative_to(ROOT))

500 accounts · 2023-01-01 to 2023-01-30
data/results/medium/2022-12-31/submission.csv


## Conclusion

In both studies, we selected the calibrated Chronos quantile ensemble at the shared ±€1,000 cap. We apply that policy to each account in the January export. In S4, we compare the matched forecasts and assess group-specific caps.